In [1]:
# ============================================================
# TrustSyn — CatBoost + D-MPNN Stacking
# Cell 1 — Environment & Prediction Discovery
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("TRUSTSYN — CATBOOST + D-MPNN STACKING")
print("=" * 70)

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/Users/anoushka/TrustSyn")

SPLITS_ROOT = PROJECT_ROOT / "splits"

CATBOOST_ROOT = (
    PROJECT_ROOT / "output" / "catboost_output"
)

DMPNN_ROOT = (
    PROJECT_ROOT / "output" / "dmpnn_output" / "FINAL_V2_C"
)

STACKING_ROOT = (
    PROJECT_ROOT / "output" / "stacking_output"
)

STACKING_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject:")
print(PROJECT_ROOT)

print("\nCatBoost:")
print(CATBOOST_ROOT)

print("\nD-MPNN:")
print(DMPNN_ROOT)

print("\nStacking output:")
print(STACKING_ROOT)

# ------------------------------------------------------------
# VERIFY CORE DIRECTORIES
# ------------------------------------------------------------

assert PROJECT_ROOT.exists(), (
    f"Project root not found: {PROJECT_ROOT}"
)

assert SPLITS_ROOT.exists(), (
    f"Splits directory not found: {SPLITS_ROOT}"
)

assert CATBOOST_ROOT.exists(), (
    f"CatBoost directory not found: {CATBOOST_ROOT}"
)

assert DMPNN_ROOT.exists(), (
    f"D-MPNN directory not found: {DMPNN_ROOT}"
)

# ------------------------------------------------------------
# FIND ALL CATBOOST CSV FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATBOOST CSV FILES")
print("=" * 70)

catboost_csvs = sorted(
    CATBOOST_ROOT.rglob("*.csv")
)

for path in catboost_csvs:
    print(
        f"{path.relative_to(CATBOOST_ROOT)}"
    )

print(
    f"\nTotal CatBoost CSV files: "
    f"{len(catboost_csvs)}"
)

# ------------------------------------------------------------
# FIND ALL D-MPNN PREDICTION FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("D-MPNN PREDICTION FILES")
print("=" * 70)

dmpnn_csvs = sorted(
    DMPNN_ROOT.rglob("*predictions*.csv")
)

for path in dmpnn_csvs:
    print(
        f"{path.relative_to(DMPNN_ROOT)}"
    )

print(
    f"\nTotal D-MPNN prediction files: "
    f"{len(dmpnn_csvs)}"
)

# ------------------------------------------------------------
# FIND CANONICAL SPLITS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CANONICAL SPLITS")
print("=" * 70)

SPLIT_DIRS = {
    "RANDOM": SPLITS_ROOT / "random",
    "COLD_COMBINATION": SPLITS_ROOT / "cold_combination",
    "COLD_CELL_LINE": SPLITS_ROOT / "cold_cell_line",
    "COLD_DRUG": SPLITS_ROOT / "cold_drug",
}

for split_name, split_dir in SPLIT_DIRS.items():

    print(f"\n{split_name}")
    print(f"  Directory: {split_dir}")

    for subset in ["train", "val", "test"]:

        path = split_dir / f"{subset}.csv"

        print(
            f"  {subset:5}: "
            f"{'FOUND' if path.exists() else 'MISSING'}"
        )

# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DISCOVERY COMPLETE")
print("=" * 70)

print(
    "\nNo model training performed."
    "\nNo prediction files modified."
    "\nNo split files modified."
    "\nNo feature files modified."
)

print(
    "\nNext: identify the exact CatBoost prediction files "
    "and match them to the canonical splits."
)

TRUSTSYN — CATBOOST + D-MPNN STACKING

Project:
/Users/anoushka/TrustSyn

CatBoost:
/Users/anoushka/TrustSyn/output/catboost_output

D-MPNN:
/Users/anoushka/TrustSyn/output/dmpnn_output/FINAL_V2_C

Stacking output:
/Users/anoushka/TrustSyn/output/stacking_output

CATBOOST CSV FILES

Total CatBoost CSV files: 0

D-MPNN PREDICTION FILES
COLD_CELL_LINE/predictions/final_v2_c_test_predictions.csv
COLD_COMBINATION/predictions/final_v2_c_test_predictions.csv
COLD_DRUG/predictions/final_v2_c_test_predictions.csv
RANDOM/predictions/final_v2_c_test_predictions.csv

Total D-MPNN prediction files: 4

CANONICAL SPLITS

RANDOM
  Directory: /Users/anoushka/TrustSyn/splits/random
  train: FOUND
  val  : FOUND
  test : FOUND

COLD_COMBINATION
  Directory: /Users/anoushka/TrustSyn/splits/cold_combination
  train: FOUND
  val  : FOUND
  test : FOUND

COLD_CELL_LINE
  Directory: /Users/anoushka/TrustSyn/splits/cold_cell_line
  train: FOUND
  val  : FOUND
  test : FOUND

COLD_DRUG
  Directory: /Users/anou

In [2]:
# ============================================================
# TrustSyn — CatBoost Prediction Artifact Search
# Cell 2
# ============================================================

print("=" * 70)
print("SEARCHING FOR CATBOOST PREDICTION ARTIFACTS")
print("=" * 70)

OUTPUT_ROOT = PROJECT_ROOT / "output"

# ------------------------------------------------------------
# SEARCH ALL FILES
# ------------------------------------------------------------

all_files = sorted(
    p for p in OUTPUT_ROOT.rglob("*")
    if p.is_file()
)

print(f"\nTotal files under output/: {len(all_files)}")

# ------------------------------------------------------------
# SEARCH BY RELEVANT TERMS
# ------------------------------------------------------------

keywords = [
    "pred",
    "prediction",
    "catboost",
    "random",
    "cold",
    "test",
]

matches = []

for path in all_files:

    name = path.name.lower()

    if any(
        keyword in name
        for keyword in keywords
    ):
        matches.append(path)

# ------------------------------------------------------------
# PRINT MATCHES
# ------------------------------------------------------------

print("\nPotential prediction/model artifacts:")

for path in matches:

    try:
        size_mb = path.stat().st_size / (1024 ** 2)
    except Exception:
        size_mb = 0

    print(
        f"{path.relative_to(OUTPUT_ROOT)}"
        f" | {size_mb:.2f} MB"
    )

print(
    f"\nPotential matches: {len(matches)}"
)

# ------------------------------------------------------------
# SEARCH SPECIFIC FILE TYPES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILE TYPE SUMMARY")
print("=" * 70)

extensions = {}

for path in all_files:

    suffix = path.suffix.lower()

    extensions[suffix] = (
        extensions.get(suffix, 0) + 1
    )

for suffix, count in sorted(
    extensions.items(),
    key=lambda x: (-x[1], x[0])
):

    print(f"{suffix or '[no extension]':10} {count}")

# ------------------------------------------------------------
# LOOK FOR CATBOOST MODEL FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATBOOST MODEL FILES")
print("=" * 70)

catboost_models = [
    p for p in all_files
    if (
        "catboost" in p.name.lower()
        or p.suffix.lower() == ".cbm"
    )
]

for path in catboost_models:

    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{path.relative_to(OUTPUT_ROOT)}"
        f" | {size_mb:.2f} MB"
    )

print(
    f"\nCatBoost model files found: "
    f"{len(catboost_models)}"
)

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)

print(
    "\nNo files modified."
    "\nNo models trained."
    "\nNo predictions generated."
)

SEARCHING FOR CATBOOST PREDICTION ARTIFACTS

Total files under output/: 202

Potential prediction/model artifacts:
catboost_output/COLD_CELL/catboost_seed_123.cbm | 2.13 MB
catboost_output/COLD_CELL/catboost_seed_2024.cbm | 2.13 MB
catboost_output/COLD_CELL/catboost_seed_3407.cbm | 2.13 MB
catboost_output/COLD_CELL/catboost_seed_42.cbm | 2.13 MB
catboost_output/COLD_CELL/catboost_seed_7777.cbm | 2.13 MB
catboost_output/COLD_COMBINATION/catboost_seed_123.cbm | 2.13 MB
catboost_output/COLD_COMBINATION/catboost_seed_2024.cbm | 2.13 MB
catboost_output/COLD_COMBINATION/catboost_seed_3407.cbm | 2.13 MB
catboost_output/COLD_COMBINATION/catboost_seed_42.cbm | 2.13 MB
catboost_output/COLD_COMBINATION/catboost_seed_7777.cbm | 2.13 MB
catboost_output/COLD_DRUG/catboost_seed_123.cbm | 2.13 MB
catboost_output/COLD_DRUG/catboost_seed_2024.cbm | 2.13 MB
catboost_output/COLD_DRUG/catboost_seed_3407.cbm | 2.13 MB
catboost_output/COLD_DRUG/catboost_seed_42.cbm | 2.13 MB
catboost_output/COLD_DRUG/catboos

In [3]:
# ================================================================
# TRUSTSYN — CATBOOST STACKING PRE-FLIGHT
# Inspect existing CatBoost models + feature objects
# ================================================================

from pathlib import Path
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/anoushka/TrustSyn")
CATBOOST_DIR = PROJECT_ROOT / "output" / "catboost_output"
DMPNN_DIR = PROJECT_ROOT / "output" / "dmpnn_output" / "FINAL_V2_C"

print("=" * 70)
print("TRUSTSYN — CATBOOST STACKING PRE-FLIGHT")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Locate CatBoost models
# ---------------------------------------------------------------

catboost_models = {}

for split_name in [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL",
    "COLD_DRUG",
]:
    split_dir = CATBOOST_DIR / split_name

    models = sorted(split_dir.glob("catboost_seed_*.cbm"))

    print(f"\n{split_name}")
    print(f"  Models found: {len(models)}")

    for model_path in models:
        print(f"    {model_path.name}")

    assert len(models) == 5, (
        f"{split_name}: expected 5 CatBoost seed models, "
        f"found {len(models)}"
    )

    catboost_models[split_name] = models

# ---------------------------------------------------------------
# 2. Locate D-MPNN test predictions
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("D-MPNN PREDICTIONS")
print("=" * 70)

dmpnn_predictions = {}

for split_name in [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL_LINE",
    "COLD_DRUG",
]:

    path = (
        DMPNN_DIR
        / split_name
        / "predictions"
        / "final_v2_c_test_predictions.csv"
    )

    print(f"\n{split_name}")
    print(f"  Path: {path}")
    print(f"  Exists: {path.exists()}")

    assert path.exists(), f"Missing D-MPNN predictions: {path}"

    df = pd.read_csv(path)

    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")

    dmpnn_predictions[split_name] = df

# ---------------------------------------------------------------
# 3. Inspect current notebook variables
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("CATBOOST FEATURE OBJECTS IN MEMORY")
print("=" * 70)

candidate_names = [
    "FINAL_FEATURES",
    "X",
    "X_train",
    "X_val",
    "X_test",
    "X_train_final",
    "X_val_final",
    "X_test_final",
    "FEATURE_MATRIX",
    "FEATURES",
    "feature_matrix",
    "master_features",
    "MASTER_FEATURES",
    "MASTER",
]

found = []

for name in candidate_names:
    if name in globals():
        obj = globals()[name]
        found.append(name)

        try:
            print(
                f"{name:<25} "
                f"type={type(obj).__name__} "
                f"shape={getattr(obj, 'shape', 'N/A')}"
            )
        except Exception:
            print(
                f"{name:<25} "
                f"type={type(obj).__name__}"
            )

if not found:
    print("No obvious CatBoost feature matrix found in memory.")

# ---------------------------------------------------------------
# 4. Inspect split DataFrames
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT DATAFRAMES IN MEMORY")
print("=" * 70)

split_candidates = [
    "FINAL_SPLITS",
    "RANDOM_TRAIN",
    "RANDOM_VAL",
    "RANDOM_TEST",
    "COLD_COMBINATION_TRAIN",
    "COLD_COMBINATION_VAL",
    "COLD_COMBINATION_TEST",
    "COLD_CELL_TRAIN_DF",
    "COLD_CELL_VAL_DF",
    "COLD_CELL_TEST_DF",
    "COLD_DRUG_TRAIN",
    "COLD_DRUG_VAL",
    "COLD_DRUG_TEST",
]

for name in split_candidates:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            print(
                f"{name:<30} "
                f"shape={obj.shape} "
                f"columns={obj.columns.tolist()}"
            )
        elif isinstance(obj, dict):
            print(
                f"{name:<30} "
                f"dict keys={list(obj.keys())}"
            )

# ---------------------------------------------------------------
# 5. Verify D-MPNN prediction schemas
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("D-MPNN SCHEMA CHECK")
print("=" * 70)

for split_name, df in dmpnn_predictions.items():

    print(f"\n{split_name}")

    required = {"drug_A", "drug_B", "CELLNAME"}

    missing = required - set(df.columns)

    print(f"  Required ID columns missing: {missing}")

    prediction_candidates = [
        c for c in df.columns
        if "pred" in c.lower()
    ]

    target_candidates = [
        c for c in df.columns
        if "target" in c.lower()
        or "score" in c.lower()
    ]

    print(f"  Prediction columns: {prediction_candidates}")
    print(f"  Target/score columns: {target_candidates}")

# ---------------------------------------------------------------
# 6. Final status
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("PRE-FLIGHT COMPLETE")
print("=" * 70)

print("""
CatBoost models:
  ✅ 5 seeds × 4 splits found

D-MPNN predictions:
  ✅ 4 canonical test prediction files found

Training:
  ❌ NOT performed

Model files:
  ❌ NOT modified

Prediction files:
  ❌ NOT modified

Next step:
  Identify the exact 62-feature CatBoost construction used during
  FINAL_V2 training, then generate CatBoost test predictions and
  verify row-for-row alignment with D-MPNN.
""")

TRUSTSYN — CATBOOST STACKING PRE-FLIGHT

RANDOM
  Models found: 5
    catboost_seed_123.cbm
    catboost_seed_2024.cbm
    catboost_seed_3407.cbm
    catboost_seed_42.cbm
    catboost_seed_7777.cbm

COLD_COMBINATION
  Models found: 5
    catboost_seed_123.cbm
    catboost_seed_2024.cbm
    catboost_seed_3407.cbm
    catboost_seed_42.cbm
    catboost_seed_7777.cbm

COLD_CELL
  Models found: 5
    catboost_seed_123.cbm
    catboost_seed_2024.cbm
    catboost_seed_3407.cbm
    catboost_seed_42.cbm
    catboost_seed_7777.cbm

COLD_DRUG
  Models found: 5
    catboost_seed_123.cbm
    catboost_seed_2024.cbm
    catboost_seed_3407.cbm
    catboost_seed_42.cbm
    catboost_seed_7777.cbm

D-MPNN PREDICTIONS

RANDOM
  Path: /Users/anoushka/TrustSyn/output/dmpnn_output/FINAL_V2_C/RANDOM/predictions/final_v2_c_test_predictions.csv
  Exists: True
  Shape: (29408, 11)
  Columns: ['drug_A', 'drug_B', 'CELLNAME', 'tissue', 'combo_score', 'CONC1', 'CONC2', 'cellminer_cellline_id', 'nci_almanac_cellname

In [4]:
# ================================================================
# TRUSTSYN — EXTRACT EXACT CATBOOST FEATURE INTERFACE
# ================================================================

from pathlib import Path
from catboost import CatBoostRegressor

PROJECT_ROOT = Path("/Users/anoushka/TrustSyn")
CATBOOST_DIR = PROJECT_ROOT / "output" / "catboost_output"

print("=" * 70)
print("TRUSTSYN — EXTRACTING EXACT CATBOOST FEATURE INTERFACE")
print("=" * 70)

MODEL_PATH = (
    CATBOOST_DIR
    / "RANDOM"
    / "catboost_seed_42.cbm"
)

print(f"\nLoading:")
print(MODEL_PATH)

model = CatBoostRegressor()
model.load_model(str(MODEL_PATH))

# ---------------------------------------------------------------
# Model information
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print("Tree count:", model.tree_count_)
print("Feature count:", model.feature_count_)

feature_names = model.feature_names_

print("\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"{i:>3}: {name}")

# ---------------------------------------------------------------
# Check all 4 split models have identical feature interfaces
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING ALL CATBOOST MODELS")
print("=" * 70)

all_features = {}

for split_name in [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL",
    "COLD_DRUG",
]:

    path = (
        CATBOOST_DIR
        / split_name
        / "catboost_seed_42.cbm"
    )

    m = CatBoostRegressor()
    m.load_model(str(path))

    names = m.feature_names_

    all_features[split_name] = names

    print(
        f"{split_name:<20} "
        f"features={len(names)} "
        f"trees={m.tree_count_}"
    )

# ---------------------------------------------------------------
# Compare feature interfaces
# ---------------------------------------------------------------

reference = all_features["RANDOM"]

print("\n" + "=" * 70)
print("FEATURE INTERFACE CONSISTENCY")
print("=" * 70)

for split_name, names in all_features.items():

    identical = names == reference

    print(
        f"{split_name:<20} "
        f"identical_to_RANDOM={identical}"
    )

    if not identical:

        print("  Differences:")

        max_len = max(
            len(reference),
            len(names)
        )

        for i in range(max_len):

            ref = (
                reference[i]
                if i < len(reference)
                else "<missing>"
            )

            cur = (
                names[i]
                if i < len(names)
                else "<missing>"
            )

            if ref != cur:
                print(
                    f"    position {i}: "
                    f"RANDOM={ref} | "
                    f"{split_name}={cur}"
                )

# ---------------------------------------------------------------
# Save feature interface ONLY for inspection
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Expected CatBoost features: {len(reference)}")

assert len(reference) == 62, (
    f"Expected 62 features, but model reports {len(reference)}"
)

print("✅ CatBoost expects exactly 62 features")
print("✅ Feature order extracted from trained model")
print("\nNo training performed.")
print("No model files modified.")
print("No split files modified.")
print("No prediction files modified.")
print("=" * 70)

TRUSTSYN — EXTRACTING EXACT CATBOOST FEATURE INTERFACE

Loading:
/Users/anoushka/TrustSyn/output/catboost_output/RANDOM/catboost_seed_42.cbm

MODEL INFORMATION
Tree count: 2000


AttributeError: 'CatBoostRegressor' object has no attribute 'feature_count_'

In [5]:
# ================================================================
# TRUSTSYN — EXTRACT EXACT CATBOOST FEATURE INTERFACE
# CORRECTED VERSION
# ================================================================

from pathlib import Path
from catboost import CatBoostRegressor

PROJECT_ROOT = Path("/Users/anoushka/TrustSyn")
CATBOOST_DIR = PROJECT_ROOT / "output" / "catboost_output"

print("=" * 70)
print("TRUSTSYN — EXTRACTING EXACT CATBOOST FEATURE INTERFACE")
print("=" * 70)

MODEL_PATH = (
    CATBOOST_DIR
    / "RANDOM"
    / "catboost_seed_42.cbm"
)

print(f"\nLoading:")
print(MODEL_PATH)

model = CatBoostRegressor()
model.load_model(str(MODEL_PATH))

# ---------------------------------------------------------------
# Model information
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print("Tree count:", model.tree_count_)

feature_names = model.feature_names_

print("Feature count:", len(feature_names))

print("\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"{i:>3}: {name}")

# ---------------------------------------------------------------
# Check all 4 split models
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING ALL CATBOOST MODELS")
print("=" * 70)

all_features = {}

for split_name in [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL",
    "COLD_DRUG",
]:

    path = (
        CATBOOST_DIR
        / split_name
        / "catboost_seed_42.cbm"
    )

    m = CatBoostRegressor()
    m.load_model(str(path))

    names = m.feature_names_

    all_features[split_name] = names

    print(
        f"{split_name:<20} "
        f"features={len(names)} "
        f"trees={m.tree_count_}"
    )

# ---------------------------------------------------------------
# Compare feature interfaces
# ---------------------------------------------------------------

reference = all_features["RANDOM"]

print("\n" + "=" * 70)
print("FEATURE INTERFACE CONSISTENCY")
print("=" * 70)

for split_name, names in all_features.items():

    identical = names == reference

    print(
        f"{split_name:<20} "
        f"identical_to_RANDOM={identical}"
    )

    if not identical:

        print("  Differences:")

        max_len = max(
            len(reference),
            len(names)
        )

        for i in range(max_len):

            ref = (
                reference[i]
                if i < len(reference)
                else "<missing>"
            )

            cur = (
                names[i]
                if i < len(names)
                else "<missing>"
            )

            if ref != cur:
                print(
                    f"    position {i}: "
                    f"RANDOM={ref} | "
                    f"{split_name}={cur}"
                )

# ---------------------------------------------------------------
# Final validation
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Expected CatBoost features: {len(reference)}")

if len(reference) == 62:
    print("✅ CatBoost expects exactly 62 features")
else:
    print(
        f"⚠️ CatBoost model reports {len(reference)} features, "
        "not 62."
    )

print("\nNo training performed.")
print("No model files modified.")
print("No split files modified.")
print("No prediction files modified.")
print("=" * 70)

TRUSTSYN — EXTRACTING EXACT CATBOOST FEATURE INTERFACE

Loading:
/Users/anoushka/TrustSyn/output/catboost_output/RANDOM/catboost_seed_42.cbm

MODEL INFORMATION
Tree count: 2000
Feature count: 62

Feature names:
  0: CellMiner_PC1
  1: CellMiner_PC2
  2: CellMiner_PC3
  3: CellMiner_PC4
  4: CellMiner_PC5
  5: CellMiner_PC6
  6: CellMiner_PC7
  7: CellMiner_PC8
  8: CellMiner_PC9
  9: CellMiner_PC10
 10: CellMiner_PC11
 11: CellMiner_PC12
 12: CellMiner_PC13
 13: CellMiner_PC14
 14: CellMiner_PC15
 15: CellMiner_PC16
 16: CellMiner_PC17
 17: CellMiner_PC18
 18: CellMiner_PC19
 19: CellMiner_PC20
 20: CellMiner_PC21
 21: CellMiner_PC22
 22: CellMiner_PC23
 23: CellMiner_PC24
 24: CellMiner_PC25
 25: CellMiner_PC26
 26: CellMiner_PC27
 27: CellMiner_PC28
 28: CellMiner_PC29
 29: CellMiner_PC30
 30: CellMiner_PC31
 31: CellMiner_PC32
 32: CellMiner_PC33
 33: CellMiner_PC34
 34: CellMiner_PC35
 35: CellMiner_PC36
 36: CellMiner_PC37
 37: CellMiner_PC38
 38: CellMiner_PC39
 39: CellMiner_PC4

In [6]:
# ================================================================
# TRUSTSYN — LOCATE EXACT 62 CATBOOST FEATURE SOURCES
# ================================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/anoushka/TrustSyn")

print("=" * 70)
print("TRUSTSYN — LOCATING CATBOOST 62-FEATURE SOURCES")
print("=" * 70)

# ---------------------------------------------------------------
# Expected CatBoost interface
# ---------------------------------------------------------------

EXPECTED_FEATURES = (
    [f"CellMiner_PC{i}" for i in range(1, 51)]
    + [
        "STRING_distance",
        "STRING_available",
        "KEGG_overlap",
        "Tanimoto_similarity",
        "target_count_A",
        "target_count_B",
        "shared_target_count",
        "union_target_count",
        "target_jaccard",
        "target_overlap_A",
        "target_overlap_B",
        "has_shared_target",
    ]
)

print("\nExpected features:", len(EXPECTED_FEATURES))

# ---------------------------------------------------------------
# Search processed feature files
# ---------------------------------------------------------------

SEARCH_ROOTS = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "interim",
]

matches = []

print("\n" + "=" * 70)
print("SEARCHING FEATURE FILES")
print("=" * 70)

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in [
            ".csv",
            ".parquet",
            ".pkl",
            ".pickle",
        ]:
            continue

        try:

            if path.suffix.lower() == ".csv":
                df_sample = pd.read_csv(
                    path,
                    nrows=5
                )

            elif path.suffix.lower() == ".parquet":
                df_sample = pd.read_parquet(
                    path
                ).head(5)

            else:
                continue

            columns = set(df_sample.columns)

            overlap = [
                f for f in EXPECTED_FEATURES
                if f in columns
            ]

            if overlap:

                matches.append(
                    (
                        path,
                        len(overlap),
                        overlap,
                        list(df_sample.columns),
                    )
                )

        except Exception:
            pass

# ---------------------------------------------------------------
# Report matches
# ---------------------------------------------------------------

matches.sort(
    key=lambda x: x[1],
    reverse=True
)

print(f"\nFiles containing expected features: {len(matches)}")

for path, count, overlap, columns in matches:

    print(
        f"\n{path.relative_to(PROJECT_ROOT)}"
    )

    print(
        f"  Matching features: "
        f"{count}/{len(EXPECTED_FEATURES)}"
    )

    if count <= 15:
        print("  Features:")
        for feature in overlap:
            print(f"    - {feature}")

# ---------------------------------------------------------------
# Identify complete feature files
# ---------------------------------------------------------------

complete = [
    x for x in matches
    if x[1] == len(EXPECTED_FEATURES)
]

print("\n" + "=" * 70)
print("COMPLETE 62-FEATURE FILES")
print("=" * 70)

if complete:

    for path, count, overlap, columns in complete:

        print(
            f"✅ {path.relative_to(PROJECT_ROOT)}"
        )

else:

    print(
        "⚠️ No single file contains all 62 features."
    )

    print(
        "This is okay if the features are distributed "
        "across multiple existing files."
    )

print("\n" + "=" * 70)
print("SAFETY CHECK")
print("=" * 70)

print("MASTER modified:        NO")
print("Splits modified:        NO")
print("Feature files modified: NO")
print("Models modified:        NO")
print("Predictions generated:  NO")

print("\nNext: use the discovered files to reconstruct")
print("the exact 62-feature CatBoost matrix.")
print("=" * 70)

TRUSTSYN — LOCATING CATBOOST 62-FEATURE SOURCES

Expected features: 62

SEARCHING FEATURE FILES

Files containing expected features: 3

data/processed/string/master_with_string_distance.csv
  Matching features: 1/62
  Features:
    - STRING_distance

data/processed/string/drug_pair_string_distance.csv
  Matching features: 1/62
  Features:
    - STRING_distance

data/interim/drug_pair_kegg_features_105drug_original.csv
  Matching features: 1/62
  Features:
    - KEGG_overlap

COMPLETE 62-FEATURE FILES
⚠️ No single file contains all 62 features.
This is okay if the features are distributed across multiple existing files.

SAFETY CHECK
MASTER modified:        NO
Splits modified:        NO
Feature files modified: NO
Models modified:        NO
Predictions generated:  NO

Next: use the discovered files to reconstruct
the exact 62-feature CatBoost matrix.


In [7]:
# ================================================================
# TRUSTSYN — FULL FEATURE SOURCE DISCOVERY
# ================================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/anoushka/TrustSyn")

EXPECTED_GROUPS = {
    "CellMiner_PC": [f"CellMiner_PC{i}" for i in range(1, 51)],
    "STRING": [
        "STRING_distance",
        "STRING_available",
    ],
    "KEGG": [
        "KEGG_overlap",
    ],
    "Drug_pair": [
        "Tanimoto_similarity",
    ],
    "Target": [
        "target_count_A",
        "target_count_B",
        "shared_target_count",
        "union_target_count",
        "target_jaccard",
        "target_overlap_A",
        "target_overlap_B",
        "has_shared_target",
    ],
}

ALL_EXPECTED = [
    x
    for group in EXPECTED_GROUPS.values()
    for x in group
]

print("=" * 70)
print("TRUSTSYN — FULL FEATURE SOURCE DISCOVERY")
print("=" * 70)

print("\nExpected feature groups:")
for group, features in EXPECTED_GROUPS.items():
    print(f"  {group:<15}: {len(features)}")

# ---------------------------------------------------------------
# Search entire TrustSyn project
# ---------------------------------------------------------------

results = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    # Ignore model/checkpoint files
    if path.suffix.lower() not in [
        ".csv",
        ".parquet",
        ".pkl",
        ".pickle",
        ".xlsx",
    ]:
        continue

    try:

        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=3)

        elif path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path).head(3)

        elif path.suffix.lower() in [".pkl", ".pickle"]:
            df = pd.read_pickle(path).head(3)

        elif path.suffix.lower() == ".xlsx":
            # Only inspect sheet names first
            xls = pd.ExcelFile(path)

            for sheet in xls.sheet_names:

                try:
                    df = pd.read_excel(
                        path,
                        sheet_name=sheet,
                        nrows=3
                    )

                    columns = set(df.columns)

                    overlap = [
                        f for f in ALL_EXPECTED
                        if f in columns
                    ]

                    if overlap:
                        results.append(
                            (
                                path,
                                sheet,
                                overlap,
                                list(df.columns)
                            )
                        )

                except Exception:
                    pass

            continue

        else:
            continue

        columns = set(df.columns)

        overlap = [
            f for f in ALL_EXPECTED
            if f in columns
        ]

        if overlap:
            results.append(
                (
                    path,
                    None,
                    overlap,
                    list(df.columns)
                )
            )

    except Exception:
        continue

# ---------------------------------------------------------------
# Display results
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FILES CONTAINING EXPECTED CATBOOST FEATURES")
print("=" * 70)

if not results:

    print("❌ No matching files found.")

else:

    for path, sheet, overlap, columns in sorted(
        results,
        key=lambda x: len(x[2]),
        reverse=True
    ):

        try:
            relative = path.relative_to(PROJECT_ROOT)
        except:
            relative = path

        print(
            f"\n{relative}"
            + (f" [sheet={sheet}]" if sheet else "")
        )

        print(
            f"  Matching: "
            f"{len(overlap)}/{len(ALL_EXPECTED)}"
        )

        for feature in overlap:
            print(f"    ✓ {feature}")

# ---------------------------------------------------------------
# Group coverage
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE GROUP COVERAGE")
print("=" * 70)

for group, features in EXPECTED_GROUPS.items():

    found = set()

    for _, _, overlap, _ in results:
        found.update(
            f for f in overlap
            if f in features
        )

    print(
        f"\n{group}: "
        f"{len(found)}/{len(features)} found"
    )

    missing = [
        f for f in features
        if f not in found
    ]

    if missing:
        print("  Missing:")
        for f in missing:
            print(f"    - {f}")

# ---------------------------------------------------------------
# Filename-based search
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FILENAME-BASED CANDIDATES")
print("=" * 70)

keywords = [
    "cellminer",
    "pca",
    "string",
    "kegg",
    "target",
    "tanimoto",
    "drug_pair",
    "drug",
    "feature",
]

seen = set()

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()

    if any(k in name for k in keywords):

        if path in seen:
            continue

        seen.add(path)

        try:
            relative = path.relative_to(PROJECT_ROOT)
        except:
            relative = path

        print(relative)

print("\n" + "=" * 70)
print("SAFETY CHECK")
print("=" * 70)

print("MASTER modified:        NO")
print("Splits modified:        NO")
print("Feature files modified: NO")
print("Models modified:        NO")
print("Predictions generated:  NO")

print("=" * 70)

TRUSTSYN — FULL FEATURE SOURCE DISCOVERY

Expected feature groups:
  CellMiner_PC   : 50
  STRING         : 2
  KEGG           : 1
  Drug_pair      : 1
  Target         : 8

FILES CONTAINING EXPECTED CATBOOST FEATURES

splits/cold_drug_corrected/val.csv
  Matching: 62/62
    ✓ CellMiner_PC1
    ✓ CellMiner_PC2
    ✓ CellMiner_PC3
    ✓ CellMiner_PC4
    ✓ CellMiner_PC5
    ✓ CellMiner_PC6
    ✓ CellMiner_PC7
    ✓ CellMiner_PC8
    ✓ CellMiner_PC9
    ✓ CellMiner_PC10
    ✓ CellMiner_PC11
    ✓ CellMiner_PC12
    ✓ CellMiner_PC13
    ✓ CellMiner_PC14
    ✓ CellMiner_PC15
    ✓ CellMiner_PC16
    ✓ CellMiner_PC17
    ✓ CellMiner_PC18
    ✓ CellMiner_PC19
    ✓ CellMiner_PC20
    ✓ CellMiner_PC21
    ✓ CellMiner_PC22
    ✓ CellMiner_PC23
    ✓ CellMiner_PC24
    ✓ CellMiner_PC25
    ✓ CellMiner_PC26
    ✓ CellMiner_PC27
    ✓ CellMiner_PC28
    ✓ CellMiner_PC29
    ✓ CellMiner_PC30
    ✓ CellMiner_PC31
    ✓ CellMiner_PC32
    ✓ CellMiner_PC33
    ✓ CellMiner_PC34
    ✓ CellMiner_PC35
  